# VeriGap — full container-free harness (V / M / R over 165 tasks)

**VeriGap: Coding Agents Build Software They Cannot Test.** Given only a program *specification*, can a model write a test suite that catches subtly-broken implementations?

This notebook is the self-contained, Docker-free harness for the three paper metrics:
- **Validity V** — fraction of generated tests that pass the reference implementation.
- **Mutation score M** — fraction of requirement-targeted mutants killed by valid tests (the headline metric).
- **Reference pass rate R** — fraction of reference tests an agent's *implementation* passes (code-gen axis).

**Data** (attached): `verigap-specs` (165 specs) + `verigap-data` (ground_truth + task defs). Reference binaries & mutants stream from the public `eth-sri/tsb-*` HF datasets. Task binaries run natively on Kaggle (no Docker).

**Modes:** `MODE='reference'` validates the scoring pipeline with the reference suite (no model, runs anywhere). `MODE='generate'` needs an OpenAI-compatible model endpoint (or `kbench.llm` in the Benchmarks runtime).

In [ ]:
import os, sys, shutil, subprocess, tempfile, gzip, hashlib, json, stat, tarfile, time
from pathlib import Path
import xml.etree.ElementTree as ET
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub','pytest'], capture_output=True)
from huggingface_hub import snapshot_download
WS = Path('/workspace')
if not shutil.which('python'): os.symlink(shutil.which('python3'), '/usr/local/bin/python')
MODE = 'reference'   # 'reference' (no model) or 'generate' (needs a model)
N_TASKS = 5          # how many of the 165 to run here; raise toward 165 as time allows

In [ ]:
# ---- data access ----
def all_task_ids():
    return sorted({p.stem for p in Path('/kaggle/input').rglob('*.md') if '__' in p.stem})
def load_spec(task):
    for p in Path('/kaggle/input').rglob(f'{task}.md'):
        return p.read_text()
    raise FileNotFoundError(task)
def fetch_reference(task):
    d = Path(snapshot_download('eth-sri/tsb-reference_impl', repo_type='dataset', allow_patterns=f'executable/{task}/**'))
    e = d/'executable'/task/'executable'; e.chmod(0o755); return e
def fetch_manifest(task):
    r = Path(snapshot_download('eth-sri/tsb-mutants', repo_type='dataset', allow_patterns=f'{task}/mutants.json'))/task
    man = json.loads((r/'mutants.json').read_text()); return r, (man['mutants'] if isinstance(man,dict) else man)
def fetch_one_mutant(task, m):
    want = (m.get('artifact') or {}).get('path') or m['payload']
    snapshot_download('eth-sri/tsb-mutants', repo_type='dataset', allow_patterns=f'{task}/{want}')
def materialize_mutant(root, m):
    p = root/m['payload']
    if p.is_file(): return p, False
    art = m['artifact']; fd,n = tempfile.mkstemp(prefix='mut-'); os.close(fd); t = Path(n)
    with gzip.open(root/art['path'],'rb') as c, open(t,'wb') as o: shutil.copyfileobj(c,o,length=1<<20)
    assert hashlib.sha256(t.read_bytes()).hexdigest()==art['sha256']; t.chmod(t.stat().st_mode|stat.S_IXUSR); return t, True
def fetch_reference_suite(task):
    td = Path(snapshot_download('eth-sri/tsb-reference_tests', repo_type='dataset', allow_patterns=f'tests/{task}.tar.gz'))
    ex = Path(tempfile.mkdtemp())/'rs'; ex.mkdir(parents=True)
    with tarfile.open(td/'tests'/f'{task}.tar.gz') as t: t.extractall(ex)
    rs = sorted(ex.rglob('run.sh')); return rs[0].parent.parent if rs else ex

In [ ]:
# ---- container-free runner + scoring ----
def run_against(exe, suite_root):
    if WS.exists(): shutil.rmtree(WS)
    WS.mkdir(parents=True)
    shutil.copy2(exe, WS/'executable'); (WS/'executable').chmod(0o755)
    for it in suite_root.iterdir():
        (shutil.copytree if it.is_dir() else shutil.copy2)(it, WS/it.name)
    try: subprocess.run(['bash','eval/run.sh'], cwd=WS, capture_output=True, text=True, timeout=120)
    except subprocess.TimeoutExpired: pass
    res = WS/'eval'/'results.xml'; out = {}
    if res.is_file():
        for tc in ET.parse(res).getroot().iter('testcase'):
            name=(tc.get('classname','')+'::'+tc.get('name','')).strip(':'); tags={c.tag for c in tc}
            out[name]='skipped' if 'skipped' in tags else ('failed' if ({'failure','error'} & tags) else 'passed')
    return out
def score_validity(task, suite_root):
    out = run_against(fetch_reference(task), suite_root)
    return (sum(1 for s in out.values() if s=='passed')/len(out)) if out else 0.0, out
def score_mutation(task, suite_root, gt_pass, cap=20):
    root, muts = fetch_manifest(task); muts = muts[:cap]; killed=0
    for m in muts:
        try: mexe,tmp = (lambda: (fetch_one_mutant(task,m) or materialize_mutant(root,m)))()
        except Exception: continue
        try:
            o = run_against(mexe, suite_root)
            if any(o.get(n)!='passed' for n in gt_pass): killed+=1
        finally:
            if tmp: Path(mexe).unlink(missing_ok=True)
    return (killed/len(muts)) if muts else None, killed, len(muts)

In [ ]:
# ---- suite source: reference (no model) or model-generated ----
INSTRUCTIONS = ('You are creating a test suite under eval/ whose tests invoke /workspace/executable. '
  'Return files incl. eval/run.sh that writes JUnit XML to /workspace/eval/results.xml '
  '(cd /workspace/eval && pytest --junitxml=/workspace/eval/results.xml).')
def get_suite(task, llm=None):
    if MODE == 'reference':
        return fetch_reference_suite(task)
    # MODE == 'generate': llm.prompt(...) -> files; plug kbench.llm or an OpenAI client here
    raise NotImplementedError('set MODE="generate" and wire an llm.prompt(...) call')

In [ ]:
# ---- run over N tasks, build the V/M table ----
import pandas as pd
rows = []
for task in all_task_ids()[:N_TASKS]:
    suite = get_suite(task)
    V, out = score_validity(task, suite)
    gt_pass = {n for n,s in out.items() if s=='passed'}
    M, killed, nmut = score_mutation(task, suite, gt_pass) if gt_pass else (None,0,0)
    rows.append({'task':task,'validity':round(V,3),'mutation':None if M is None else round(M,3),'killed':killed,'mutants':nmut})
    print(rows[-1])
df = pd.DataFrame(rows); df.to_csv('/kaggle/working/verigap_results.csv', index=False)
print(df); print('means:', df[['validity','mutation']].mean(numeric_only=True).to_dict())

## Scaling to all 165 & the paper's R
- Raise `N_TASKS` toward 165 (reference mode is fast; model mode is bounded by your model throughput).
- Set `MODE='generate'` and wire a model into `get_suite` (`kbench.llm.prompt` in the Benchmarks runtime, or any OpenAI-compatible client) to score real agents.
- **R (reference pass rate)**: build the agent's implementation then `run_against(built_exe, fetch_reference_suite(task))`; only tasks whose toolchain is present will build here (the paper uses Docker for the rest).
- The leaderboard versions of these metrics are the published tasks: **verigap-validity-v2**, **verigap-mutation-v4**, **verigap-codegen**.